# Local CPU Dueling Double DQN for Flappy Bird

This notebook is the Windows/Python 3.11 local version of the original experiment. It keeps the successful learning design while making process creation, artifact paths, evaluation, checkpointing, playback, and validation explicit.

The default `RUN_MODE` is `full`. Implementation validation uses the environment variable `FLAPPY_RUN_MODE=smoke`, so the selected 10M/20M/25M budget is never started accidentally during smoke testing.

## 1. Python 3.11 setup and local paths

Open this notebook with the **Flappy Bird Python 3.11** kernel. The setup cell fails immediately under any other Python major/minor version. It also forces CPU execution and limits native thread pools before importing NumPy or PyTorch.

In [1]:
import os
import sys

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "This notebook requires Python 3.11.x. "
        f"Current interpreter: {sys.version.split()[0]} at {sys.executable}"
    )

SELECTED_N_ENVS = 16
SELECTED_TORCH_THREADS = 1
SELECTED_GRADIENT_STEPS = 5
SELECTED_TOTAL_TIMESTEPS = 10000000
MEASURED_LEARNING_TPS = 0.0
RUNTIME_ESTIMATES_HOURS = {
    10_000_000: 0.0,
    20_000_000: 0.0,
    25_000_000: 0.0,
}
CONSERVATIVE_SELECTED_RUNTIME_HOURS = 0.0

for variable in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ[variable] = str(SELECTED_TORCH_THREADS)

import copy
import io
import json
import math
import random
import shutil
import statistics
import time
from datetime import datetime
from pathlib import Path

import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import psutil
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image as PILImage
from PIL import ImageDraw
from tqdm.auto import tqdm

import flappy_bird_gymnasium  # registers FlappyBird-v0
from flappy_local_runtime import (
    close_environments,
    create_training_vec_env,
    resolve_repository_root,
)

PROJECT_ROOT = resolve_repository_root()
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"
RUN_MODE = os.environ.get("FLAPPY_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke", "benchmark"}:
    raise ValueError("FLAPPY_RUN_MODE must be full, smoke, or benchmark.")

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_SUFFIX = "" if RUN_MODE == "full" else f"_{RUN_MODE}"
RUN_DIR = ARTIFACTS_ROOT / (
    f"ddqn_local_{SELECTED_TOTAL_TIMESTEPS}_{RUN_TIMESTAMP}{RUN_SUFFIX}"
)
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MP_START_METHOD = "spawn"
DEVICE = torch.device("cpu")

torch.set_num_threads(SELECTED_TORCH_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

PHYSICAL_CORES = psutil.cpu_count(logical=False)
LOGICAL_THREADS = psutil.cpu_count(logical=True) or os.cpu_count() or 1
if PHYSICAL_CORES is None:
    PHYSICAL_CORES = max(1, LOGICAL_THREADS // 2)

print(f"Python: {sys.version.split()[0]}")
print(f"sys.executable: {sys.executable}")
print(f"Python is exactly 3.11: {sys.version_info[:2] == (3, 11)}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Gymnasium: {gym.__version__}")
print(f"Pygame: {pygame.version.ver}")
print(f"Physical CPU cores: {PHYSICAL_CORES}")
print(f"Logical CPU threads: {LOGICAL_THREADS}")
print(f"Selected training workers: {SELECTED_N_ENVS}")
print(f"Selected PyTorch threads: {SELECTED_TORCH_THREADS}")
print(f"Multiprocessing start method: {MP_START_METHOD} (explicit)")
print(f"Artifact directory: {RUN_DIR}")
print(f"Training device: {DEVICE}")

if not (14 <= PHYSICAL_CORES <= 18 and 20 <= LOGICAL_THREADS <= 28):
    print(
        "WARNING: detected CPU topology differs materially from the expected "
        "i7-13700K layout (16 physical cores, 24 logical threads). "
        "Rerun the configuration benchmark before a long training run."
    )

Python: 3.11.9
sys.executable: C:\Users\yahya\OneDrive - UMM AL-QURA UNIVERSITY\KAUST\Summer\week4\flappy_bird\.venv\Scripts\python.exe
Python is exactly 3.11: True
PyTorch: 2.13.0+cpu
NumPy: 1.26.4
Gymnasium: 1.3.0
Pygame: 2.6.1
Physical CPU cores: 14
Logical CPU threads: 20
Selected training workers: 16
Selected PyTorch threads: 1
Multiprocessing start method: spawn (explicit)
Artifact directory: C:\Users\yahya\OneDrive - UMM AL-QURA UNIVERSITY\KAUST\Summer\week4\flappy_bird\artifacts\ddqn_local_10000000_20260726_113455_benchmark
Training device: cpu


## 2. Algorithm and training configuration

- **Dueling DQN** learns a state-value stream and a per-action advantage stream, then combines them into Q-values.
- **Double DQN** selects the next action with the online network and evaluates that action with the target network, reducing overestimation.
- **Prioritized Experience Replay (PER)** samples transitions in proportion to TD-error-derived priorities and corrects the bias with importance weights.
- **Biased epsilon-greedy exploration** does not choose flap/no-flap 50/50. A flap is an impulse that should be rare, so exploratory decisions flap with probability 0.08.
- **Reward shaping** is training-only: survival and passing pipes are rewarded, while death and upper-boundary failures are penalized.
- Evaluation uses the unmodified environment, true `info["score"]`, and greedy `epsilon=0.0` actions.

The original three-worker run used one gradient update per three collected transitions. The selected local `GRADIENT_STEPS` keeps the ratio near one update per 3–4 transitions even with many workers.

In [2]:
SEED = 42
PIPE_GAP = 100
TRAINING_SCORE_LIMIT = 1_000_000

TOTAL_TIMESTEPS = SELECTED_TOTAL_TIMESTEPS
SMOKE_TOTAL_TIMESTEPS = 25_000
RUN_TIMESTEPS = SMOKE_TOTAL_TIMESTEPS if RUN_MODE == "smoke" else TOTAL_TIMESTEPS

EVAL_INTERVAL = 250_000
CHECKPOINT_INTERVAL = 250_000
TARGET_UPDATE_INTERVAL = 10_000

BUFFER_SIZE = 500_000
LEARNING_STARTS = 20_000
BATCH_SIZE = 256
GAMMA = 0.99
LEARNING_RATE = 1e-4
HIDDEN_DIM = 192

TRAIN_EVERY_VECTOR_STEPS = 1
GRADIENT_STEPS = SELECTED_GRADIENT_STEPS
APPROX_TRANSITIONS_PER_UPDATE = SELECTED_N_ENVS / GRADIENT_STEPS

EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY_STEPS = 400_000
EXPLORATION_FLAP_PROBABILITY = 0.08

PER_ALPHA = 0.3
PER_BETA_START = 0.4
PER_BETA_END = 1.0
PER_BETA_STEPS = TOTAL_TIMESTEPS

SURVIVAL_REWARD = 0.05
PIPE_REWARD = 5.0
DEATH_PENALTY = -5.0
TOP_PENALTY = -1.0

FINAL_HOLDOUT_SEEDS = [1_000_001, 1_000_002, 1_000_003, 1_000_004, 1_000_005]
BENCHMARK_SEED = 700_000
SMOKE_EVALUATION_SEEDS = [800_001, 800_002, 800_003, 800_004, 800_005]
GIF_SEED = 900_001
DEFAULT_LIVE_SEED = 950_001

RESUME_CHECKPOINT = None  # Set to a checkpoints/ddqn_*.pt path to resume.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Run mode: {RUN_MODE}")
print(f"Run transition budget: {RUN_TIMESTEPS:,}")
print(f"N_ENVS: {SELECTED_N_ENVS}")
print(f"PyTorch threads: {SELECTED_TORCH_THREADS}")
print(f"Gradient steps per vector step: {GRADIENT_STEPS}")
print(f"Approximate update-to-data ratio: 1:{APPROX_TRANSITIONS_PER_UPDATE:.2f}")
print(f"Measured learning throughput: {MEASURED_LEARNING_TPS:,.1f} transitions/s")
print(f"Epsilon: {EPSILON_START} -> {EPSILON_END} over {EPSILON_DECAY_STEPS:,} transitions")
print(f"Periodic evaluation interval: {EVAL_INTERVAL:,} transitions")

Run mode: benchmark
Run transition budget: 10,000,000
N_ENVS: 16
PyTorch threads: 1
Gradient steps per vector step: 5
Approximate update-to-data ratio: 1:3.20
Measured learning throughput: 0.0 transitions/s
Epsilon: 1.0 -> 0.01 over 400,000 transitions
Periodic evaluation interval: 250,000 transitions


## 3. Replay buffer and Dueling Q-network

The sum tree stores proportional PER priorities in \(O(\log N)\) update and sampling time. Replay contents are intentionally not included in checkpoints because a 500,000-transition replay snapshot would make frequent checkpoints very large.

In [3]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity, state_dim, alpha=0.3, priority_epsilon=1e-5):
        self.capacity = int(capacity)
        self.state_dim = int(state_dim)
        self.alpha = float(alpha)
        self.priority_epsilon = float(priority_epsilon)
        tree_capacity = 1
        while tree_capacity < self.capacity:
            tree_capacity *= 2
        self.tree_capacity = tree_capacity
        self.priority_tree = np.zeros(2 * tree_capacity, dtype=np.float32)
        self.states = np.empty((self.capacity, self.state_dim), dtype=np.float32)
        self.next_states = np.empty_like(self.states)
        self.actions = np.empty(self.capacity, dtype=np.int64)
        self.rewards = np.empty(self.capacity, dtype=np.float32)
        self.dones = np.empty(self.capacity, dtype=np.float32)
        self.position = 0
        self.size = 0
        self.max_priority = 1.0

    def __len__(self):
        return self.size

    def _set_priority(self, data_index, priority):
        tree_index = self.tree_capacity + int(data_index)
        change = float(priority) - float(self.priority_tree[tree_index])
        while tree_index >= 1:
            self.priority_tree[tree_index] += change
            tree_index //= 2

    def add(self, state, action, reward, next_state, done):
        index = self.position
        self.states[index] = np.asarray(state, dtype=np.float32)
        self.actions[index] = int(action)
        self.rewards[index] = float(reward)
        self.next_states[index] = np.asarray(next_state, dtype=np.float32)
        self.dones[index] = float(done)
        self._set_priority(index, self.max_priority)
        self.position = (self.position + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def add_batch(self, states, actions, rewards, next_states, dones):
        for index in range(len(actions)):
            self.add(
                states[index],
                actions[index],
                rewards[index],
                next_states[index],
                dones[index],
            )

    def _find_prefix_sum_index(self, mass):
        tree_index = 1
        while tree_index < self.tree_capacity:
            left = 2 * tree_index
            if mass < self.priority_tree[left]:
                tree_index = left
            else:
                mass -= self.priority_tree[left]
                tree_index = left + 1
        return min(tree_index - self.tree_capacity, self.size - 1)

    def sample(self, batch_size, beta, device):
        if self.size < batch_size:
            raise ValueError("Replay buffer does not contain enough samples.")
        total_priority = float(self.priority_tree[1])
        if total_priority <= 0:
            raise RuntimeError("Replay priorities must be positive.")
        segment = total_priority / batch_size
        indices = np.empty(batch_size, dtype=np.int64)
        priorities = np.empty(batch_size, dtype=np.float32)
        for sample_index in range(batch_size):
            mass = np.random.uniform(
                segment * sample_index,
                segment * (sample_index + 1),
            )
            data_index = self._find_prefix_sum_index(mass)
            indices[sample_index] = data_index
            priorities[sample_index] = self.priority_tree[
                self.tree_capacity + data_index
            ]
        probabilities = np.maximum(priorities / total_priority, 1e-12)
        weights = (self.size * probabilities) ** (-float(beta))
        weights /= weights.max() + 1e-8
        return {
            "states": torch.as_tensor(self.states[indices], device=device),
            "actions": torch.as_tensor(self.actions[indices], device=device),
            "rewards": torch.as_tensor(self.rewards[indices], device=device),
            "next_states": torch.as_tensor(self.next_states[indices], device=device),
            "dones": torch.as_tensor(self.dones[indices], device=device),
            "weights": torch.as_tensor(weights, dtype=torch.float32, device=device),
            "indices": indices,
        }

    def update_priorities(self, indices, td_errors):
        priorities = (
            np.abs(np.asarray(td_errors, dtype=np.float32)) + self.priority_epsilon
        ) ** self.alpha
        for data_index, priority in zip(indices, priorities):
            self._set_priority(int(data_index), float(priority))
        self.max_priority = max(self.max_priority, float(priorities.max()))


class DuelingQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=192):
        super().__init__()
        self.state_dim = int(state_dim)
        self.action_dim = int(action_dim)
        self.hidden_dim = int(hidden_dim)
        self.features = nn.Sequential(
            nn.Linear(self.state_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(self.hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(self.hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, self.action_dim),
        )

    def forward(self, states):
        features = self.features(states)
        values = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return values + advantages - advantages.mean(dim=1, keepdim=True)

    @torch.no_grad()
    def choose_actions(
        self,
        states,
        epsilon=0.0,
        exploration_flap_probability=0.08,
    ):
        state_tensor = torch.as_tensor(
            np.asarray(states, dtype=np.float32),
            device=next(self.parameters()).device,
        )
        greedy_actions = self(state_tensor).argmax(dim=1).cpu().numpy()
        if epsilon <= 0:
            return greedy_actions.astype(np.int64)
        explore_mask = np.random.random(len(greedy_actions)) < epsilon
        exploratory_actions = (
            np.random.random(len(greedy_actions)) < exploration_flap_probability
        ).astype(np.int64)
        return np.where(
            explore_mask, exploratory_actions, greedy_actions
        ).astype(np.int64)

## 4. Double-DQN optimization and deterministic evaluation

Training targets use the online network for action selection and the target network for action evaluation. Evaluation environments are never reward-wrapped and never rendered during checkpoint selection.

In [4]:
def linear_schedule(step, start_value, end_value, duration):
    fraction = min(max(step / max(duration, 1), 0.0), 1.0)
    return start_value + fraction * (end_value - start_value)


def optimize_double_dqn(
    online_network,
    target_network,
    optimizer,
    replay_buffer,
    batch_size,
    gamma,
    beta,
    device=DEVICE,
):
    batch = replay_buffer.sample(batch_size, beta, device)
    q_values = online_network(batch["states"]).gather(
        1, batch["actions"].unsqueeze(1)
    ).squeeze(1)
    with torch.no_grad():
        next_actions = online_network(batch["next_states"]).argmax(
            dim=1, keepdim=True
        )
        next_q_values = target_network(batch["next_states"]).gather(
            1, next_actions
        ).squeeze(1)
        targets = batch["rewards"] + gamma * (1.0 - batch["dones"]) * next_q_values
    td_errors = targets - q_values
    per_sample_loss = F.smooth_l1_loss(q_values, targets, reduction="none")
    loss = (batch["weights"] * per_sample_loss).mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(online_network.parameters(), max_norm=10.0)
    optimizer.step()
    replay_buffer.update_priorities(
        batch["indices"], td_errors.detach().cpu().numpy()
    )
    return {
        "loss": float(loss.item()),
        "mean_q": float(q_values.detach().mean().item()),
        "mean_abs_td_error": float(td_errors.detach().abs().mean().item()),
    }


def make_evaluation_env(score_limit, render_mode=None):
    return gym.make(
        "FlappyBird-v0",
        render_mode=render_mode,
        use_lidar=False,
        normalize_obs=True,
        pipe_gap=PIPE_GAP,
        score_limit=int(score_limit),
        audio_on=False,
    )


def scaled_frame_safety_limit(score_limit):
    return max(10_000, int(score_limit) * 50 + 5_000)


@torch.no_grad()
def evaluate_dqn(
    network,
    seeds,
    score_limit,
    max_steps=None,
):
    was_training = network.training
    network.eval()
    max_steps = scaled_frame_safety_limit(score_limit) if max_steps is None else int(max_steps)
    episodes = []
    try:
        for seed in seeds:
            env = make_evaluation_env(score_limit, render_mode=None)
            try:
                observation, info = env.reset(seed=int(seed))
                total_reward = 0.0
                score = int(info.get("score", 0))
                terminated = truncated = False
                for steps in range(1, max_steps + 1):
                    action = int(
                        network.choose_actions(
                            np.expand_dims(observation, axis=0), epsilon=0.0
                        )[0]
                    )
                    observation, reward, terminated, truncated, info = env.step(action)
                    total_reward += float(reward)
                    score = int(info.get("score", score))
                    if terminated or truncated:
                        break
                else:
                    steps = max_steps
                target_reached = score >= score_limit
                if target_reached:
                    reason = "target reached"
                elif terminated:
                    reason = "terminated"
                elif truncated:
                    reason = "environment truncated"
                else:
                    reason = "safety frame limit"
                episodes.append(
                    {
                        "seed": int(seed),
                        "score": int(score),
                        "reward": float(total_reward),
                        "frames": int(steps),
                        "terminated": bool(terminated),
                        "truncated": bool(truncated),
                        "target_reached": bool(target_reached),
                        "termination_reason": reason,
                    }
                )
            finally:
                env.close()
    finally:
        if was_training:
            network.train()

    scores = [episode["score"] for episode in episodes]
    return {
        "seeds": [int(seed) for seed in seeds],
        "score_limit": int(score_limit),
        "episodes": episodes,
        "scores": scores,
        "rewards": [episode["reward"] for episode in episodes],
        "episode_lengths": [episode["frames"] for episode in episodes],
        "terminations": [episode["terminated"] for episode in episodes],
        "truncations": [episode["truncated"] for episode in episodes],
        "target_reached_flags": [
            episode["target_reached"] for episode in episodes
        ],
        "mean_score": float(np.mean(scores)),
        "median_score": float(np.median(scores)),
        "std_score": float(np.std(scores)),
        "minimum_score": int(np.min(scores)),
        "maximum_score": int(np.max(scores)),
        "reached_limit_count": int(sum(score >= score_limit for score in scores)),
    }

## 5. Dynamic five-slot periodic evaluation

Periodic validation starts with seeds `[1, 2, 3, 4, 5]` and a score limit of 1000. Every successful slot is retired and replaced by a different never-before-used seed. The limit rises by exactly 1000 once per evaluation if any slot succeeds.

The checkpoint ranking key is:

`(highest_completed_level, slots_reaching_evaluated_limit, minimum_score, mean_score, maximum_score)`

`highest_completed_level` changes only when a slot actually completes the evaluated level. Merely moving to a higher pending limit cannot replace a stronger checkpoint.

In [5]:
def initial_evaluation_state():
    return {
        "active_eval_seeds": [1, 2, 3, 4, 5],
        "next_available_eval_seed": 6,
        "used_eval_seeds": [1, 2, 3, 4, 5],
        "retired_eval_seeds": [],
        "evaluation_score_limit": 1000,
        "highest_completed_level": 0,
    }


def validate_evaluation_state(state):
    active = [int(value) for value in state["active_eval_seeds"]]
    used = {int(value) for value in state["used_eval_seeds"]}
    retired = {int(value) for value in state["retired_eval_seeds"]}
    assert len(active) == 5 and len(set(active)) == 5
    assert set(active).issubset(used)
    assert retired.issubset(used)
    assert set(active).isdisjoint(retired)
    assert int(state["next_available_eval_seed"]) not in used
    assert int(state["evaluation_score_limit"]) >= 1000
    return True


def advance_evaluation_state(state, scores):
    state_before = copy.deepcopy(state)
    validate_evaluation_state(state_before)
    if len(scores) != 5:
        raise ValueError("Exactly five periodic evaluation scores are required.")

    active = list(state_before["active_eval_seeds"])
    used = set(state_before["used_eval_seeds"])
    retired = set(state_before["retired_eval_seeds"])
    limit = int(state_before["evaluation_score_limit"])
    successful_slots = [
        index for index, score in enumerate(scores) if int(score) >= limit
    ]

    next_seed = int(state_before["next_available_eval_seed"])
    replacements = {}
    if successful_slots:
        for slot in successful_slots:
            retired.add(active[slot])
            while next_seed in used:
                next_seed += 1
            replacements[slot] = next_seed
            active[slot] = next_seed
            used.add(next_seed)
            next_seed += 1
        next_limit = limit + 1000
        highest_completed = max(
            int(state_before["highest_completed_level"]), limit
        )
    else:
        next_limit = limit
        highest_completed = int(state_before["highest_completed_level"])

    while next_seed in used:
        next_seed += 1

    state_after = {
        "active_eval_seeds": active,
        "next_available_eval_seed": next_seed,
        "used_eval_seeds": sorted(used),
        "retired_eval_seeds": sorted(retired),
        "evaluation_score_limit": next_limit,
        "highest_completed_level": highest_completed,
    }
    validate_evaluation_state(state_after)
    return {
        "state_before": state_before,
        "state_after": state_after,
        "successful_slots": successful_slots,
        "replacement_seeds_by_slot": replacements,
        "evaluated_score_limit": limit,
    }


def checkpoint_ranking_key(evaluation_record):
    scores = [int(score) for score in evaluation_record["scores"]]
    return (
        int(evaluation_record["state_after"]["highest_completed_level"]),
        int(len(evaluation_record["successful_slots"])),
        int(min(scores)),
        float(np.mean(scores)),
        int(max(scores)),
    )


def run_evaluation_state_tests():
    base = initial_evaluation_state()

    none = advance_evaluation_state(base, [999, 20, 30, 40, 50])
    assert none["state_after"] == base

    one = advance_evaluation_state(base, [1000, 45, 500, 644, 754])
    assert one["state_after"]["active_eval_seeds"] == [6, 2, 3, 4, 5]
    assert one["state_after"]["next_available_eval_seed"] == 7
    assert one["state_after"]["evaluation_score_limit"] == 2000

    multiple = advance_evaluation_state(
        one["state_after"], [2000, 350, 2000, 900, 2000]
    )
    assert multiple["state_after"]["active_eval_seeds"] == [7, 2, 8, 4, 9]
    assert multiple["state_after"]["next_available_eval_seed"] == 10
    assert multiple["state_after"]["evaluation_score_limit"] == 3000

    all_five = advance_evaluation_state(
        initial_evaluation_state(), [1000] * 5
    )
    replacements = list(all_five["replacement_seeds_by_slot"].values())
    assert len(replacements) == len(set(replacements)) == 5
    assert all_five["state_after"]["evaluation_score_limit"] == 2000
    assert set(replacements).isdisjoint({1, 2, 3, 4, 5})

    resumed = json.loads(json.dumps(multiple["state_after"]))
    assert resumed == multiple["state_after"]
    validate_evaluation_state(resumed)
    return {
        "no_success": "passed",
        "one_success": "passed",
        "multiple_successes": "passed",
        "all_successes": "passed",
        "unique_replacements": "passed",
        "single_limit_increment": "passed",
        "no_seed_reuse": "passed",
        "resume_state": "passed",
    }


EVALUATION_STATE_TEST_RESULTS = run_evaluation_state_tests()
print(json.dumps(EVALUATION_STATE_TEST_RESULTS, indent=2))

{
  "no_success": "passed",
  "one_success": "passed",
  "multiple_successes": "passed",
  "all_successes": "passed",
  "unique_replacements": "passed",
  "single_limit_increment": "passed",
  "no_seed_reuse": "passed",
  "resume_state": "passed"
}

## 6. Checkpointing and exact resume metadata

Checkpoints are written atomically and include the two networks, optimizer, global step, schedule/configuration, dynamic evaluation state, best ranking state, and next scheduled events. Replay contents are not saved. Therefore a resumed run is honest continuation of weights and metadata, but replay warmup restarts and is not identical to uninterrupted training.

In [6]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def load_torch_payload(path, device=DEVICE):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def build_model_config(state_dim, action_dim):
    return {
        "state_dim": int(state_dim),
        "action_dim": int(action_dim),
        "hidden_dim": HIDDEN_DIM,
        "learning_rate": LEARNING_RATE,
        "gamma": GAMMA,
        "buffer_size": BUFFER_SIZE,
        "batch_size": BATCH_SIZE,
        "learning_starts": LEARNING_STARTS,
        "replay_buffer_serialized": False,
        "train_every_vector_steps": TRAIN_EVERY_VECTOR_STEPS,
        "gradient_steps": GRADIENT_STEPS,
        "target_update_interval": TARGET_UPDATE_INTERVAL,
        "epsilon_start": EPSILON_START,
        "epsilon_end": EPSILON_END,
        "epsilon_decay_steps": EPSILON_DECAY_STEPS,
        "exploration_flap_probability": EXPLORATION_FLAP_PROBABILITY,
        "per_alpha": PER_ALPHA,
        "per_beta_start": PER_BETA_START,
        "per_beta_end": PER_BETA_END,
        "per_beta_steps": PER_BETA_STEPS,
        "n_envs": SELECTED_N_ENVS,
        "torch_threads": SELECTED_TORCH_THREADS,
        "approx_transitions_per_update": APPROX_TRANSITIONS_PER_UPDATE,
        "total_timesteps": TOTAL_TIMESTEPS,
        "eval_interval": EVAL_INTERVAL,
        "checkpoint_interval": CHECKPOINT_INTERVAL,
        "pipe_gap": PIPE_GAP,
        "reward_shaping": {
            "survival": SURVIVAL_REWARD,
            "pipe": PIPE_REWARD,
            "death": DEATH_PENALTY,
            "top": TOP_PENALTY,
        },
        "checkpoint_ranking_key": [
            "highest_completed_level",
            "slots_reaching_evaluated_limit",
            "minimum_score",
            "mean_score",
            "maximum_score",
        ],
    }


def save_training_checkpoint(
    path,
    online_network,
    target_network,
    optimizer,
    global_step,
    model_config,
    evaluation_state,
    best_ranking_key,
    best_checkpoint_step,
    latest_evaluation,
    next_evaluation_step,
    next_checkpoint_step,
    next_target_update_step,
    run_status,
):
    payload = {
        "algorithm": "Dueling Double DQN with Prioritized Experience Replay",
        "online_state_dict": online_network.state_dict(),
        "target_state_dict": target_network.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "global_step": int(global_step),
        "model_config": dict(model_config),
        "epsilon_schedule": {
            "start": EPSILON_START,
            "end": EPSILON_END,
            "duration_transitions": EPSILON_DECAY_STEPS,
        },
        "replay_configuration": {
            "capacity": BUFFER_SIZE,
            "alpha": PER_ALPHA,
            "beta_start": PER_BETA_START,
            "beta_end": PER_BETA_END,
            "serialized": False,
            "resume_warmup_required": True,
        },
        "evaluation_state": copy.deepcopy(evaluation_state),
        "best_ranking_key": list(best_ranking_key),
        "best_checkpoint_step": best_checkpoint_step,
        "latest_evaluation": latest_evaluation,
        "selected_worker_configuration": {
            "n_envs": SELECTED_N_ENVS,
            "torch_threads": SELECTED_TORCH_THREADS,
            "gradient_steps": GRADIENT_STEPS,
            "approx_transitions_per_update": APPROX_TRANSITIONS_PER_UPDATE,
        },
        "next_scheduled_steps": {
            "evaluation": int(next_evaluation_step),
            "checkpoint": int(next_checkpoint_step),
            "target_update": int(next_target_update_step),
        },
        "run_status": str(run_status),
    }
    atomic_torch_save(payload, path)


def create_networks_and_optimizer(state_dim, action_dim):
    online = DuelingQNetwork(state_dim, action_dim, HIDDEN_DIM).to(DEVICE)
    target = DuelingQNetwork(state_dim, action_dim, HIDDEN_DIM).to(DEVICE)
    target.load_state_dict(online.state_dict())
    target.eval()
    optimizer = torch.optim.Adam(online.parameters(), lr=LEARNING_RATE)
    return online, target, optimizer


def load_model_from_checkpoint(path):
    checkpoint = load_torch_payload(Path(path))
    config = checkpoint.get("model_config", checkpoint.get("config"))
    network = DuelingQNetwork(
        int(config["state_dim"]),
        int(config["action_dim"]),
        int(config["hidden_dim"]),
    ).to(DEVICE)
    network.load_state_dict(checkpoint["online_state_dict"])
    network.eval()
    return network, checkpoint

## 7. CPU configuration benchmark

Set `FLAPPY_RUN_MODE=benchmark` to run this bounded benchmark. Every candidate creates fresh workers, replay, networks, and optimizer; it warms replay through the real 20,000-transition `LEARNING_STARTS`, then measures environment plus DDQN learning throughput. No benchmark replay or weights are reused for training.

In [7]:
BENCHMARK_CONFIGURATIONS = [
    (12, 1),
    (12, 2),
    (12, 4),
    (16, 1),
    (16, 2),
    (20, 1),
    (20, 2),
]


def gradient_steps_for_workers(n_envs):
    return max(1, int(round(int(n_envs) / 3.25)))


def measure_evaluation_frame_rate(network, frames_to_measure=5_000):
    env = make_evaluation_env(1_000_000, render_mode=None)
    observation, _ = env.reset(seed=BENCHMARK_SEED + 50_000)
    completed = 0
    started = time.perf_counter()
    try:
        while completed < frames_to_measure:
            action = int(
                network.choose_actions(
                    np.expand_dims(observation, axis=0), epsilon=0.0
                )[0]
            )
            observation, _, terminated, truncated, _ = env.step(action)
            completed += 1
            if terminated or truncated:
                observation, _ = env.reset(
                    seed=BENCHMARK_SEED + 50_000 + completed
                )
    finally:
        env.close()
    return completed / max(time.perf_counter() - started, 1e-9)


def benchmark_configuration(n_envs, torch_threads):
    gradient_steps = gradient_steps_for_workers(n_envs)
    torch.set_num_threads(torch_threads)
    environment = None
    children_before = {
        child.pid for child in psutil.Process().children(recursive=True)
    }
    rank_counts = {rank: 0 for rank in range(n_envs)}
    cpu_samples = []
    checkpoint_seconds = float("nan")

    try:
        environment = create_training_vec_env(
            n_envs=n_envs,
            base_seed=BENCHMARK_SEED,
            score_limit=TRAINING_SCORE_LIMIT,
            pipe_gap=PIPE_GAP,
            torch_threads=torch_threads,
        )
        observations = environment.reset()
        state_dim = int(environment.observation_space.shape[0])
        action_dim = int(environment.action_space.n)
        online, target, optimizer = create_networks_and_optimizer(
            state_dim, action_dim
        )
        replay = PrioritizedReplayBuffer(
            max(LEARNING_STARTS + n_envs * 500, 30_000),
            state_dim,
            PER_ALPHA,
        )

        env_only_steps = 100
        started = time.perf_counter()
        for _ in range(env_only_steps):
            actions = (
                np.random.random(n_envs) < EXPLORATION_FLAP_PROBABILITY
            ).astype(np.int64)
            next_observations, _, _, infos = environment.step(actions)
            observations = next_observations
            for info in infos:
                rank_counts[int(info["worker_rank"])] += 1
        environment_tps = (
            env_only_steps * n_envs / max(time.perf_counter() - started, 1e-9)
        )

        collected = 0
        while collected < LEARNING_STARTS:
            actions = (
                np.random.random(n_envs) < EXPLORATION_FLAP_PROBABILITY
            ).astype(np.int64)
            next_observations, rewards, dones, infos = environment.step(actions)
            replay_next = next_observations.copy()
            for index, done in enumerate(dones):
                if done and "terminal_observation" in infos[index]:
                    replay_next[index] = infos[index]["terminal_observation"]
                rank_counts[int(infos[index]["worker_rank"])] += 1
            replay.add_batch(
                observations, actions, rewards, replay_next, dones
            )
            observations = next_observations
            collected += n_envs

        psutil.cpu_percent(percpu=True)
        measurement_vector_steps = 180
        learning_started = time.perf_counter()
        losses = []
        for vector_index in range(measurement_vector_steps):
            actions = online.choose_actions(
                observations,
                epsilon=0.1,
                exploration_flap_probability=EXPLORATION_FLAP_PROBABILITY,
            )
            next_observations, rewards, dones, infos = environment.step(actions)
            replay_next = next_observations.copy()
            for index, done in enumerate(dones):
                if done and "terminal_observation" in infos[index]:
                    replay_next[index] = infos[index]["terminal_observation"]
                rank_counts[int(infos[index]["worker_rank"])] += 1
            replay.add_batch(
                observations, actions, rewards, replay_next, dones
            )
            observations = next_observations
            for _ in range(gradient_steps):
                metrics = optimize_double_dqn(
                    online,
                    target,
                    optimizer,
                    replay,
                    BATCH_SIZE,
                    GAMMA,
                    0.4,
                )
                losses.append(metrics["loss"])
            if (vector_index + 1) % 60 == 0:
                target.load_state_dict(online.state_dict())
                cpu_samples.append(psutil.cpu_percent(percpu=True))

        learning_seconds = time.perf_counter() - learning_started
        learning_transitions = measurement_vector_steps * n_envs
        evaluation_fps = measure_evaluation_frame_rate(online)

        buffer = io.BytesIO()
        save_started = time.perf_counter()
        torch.save(
            {
                "online_state_dict": online.state_dict(),
                "target_state_dict": target.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            buffer,
        )
        checkpoint_seconds = time.perf_counter() - save_started
    finally:
        close_environments([environment])
        time.sleep(0.5)

    children_after = {
        child.pid for child in psutil.Process().children(recursive=True)
    }
    lingering = sorted(children_after - children_before)
    mean_per_core = (
        np.mean(np.asarray(cpu_samples, dtype=np.float32), axis=0)
        if cpu_samples
        else np.zeros(LOGICAL_THREADS, dtype=np.float32)
    )
    all_workers_produced = all(count > 0 for count in rank_counts.values())
    return {
        "n_envs": int(n_envs),
        "torch_threads": int(torch_threads),
        "gradient_steps": int(gradient_steps),
        "approx_transitions_per_update": float(n_envs / gradient_steps),
        "environment_transitions_per_second": float(environment_tps),
        "learning_transitions_per_second": float(
            learning_transitions / max(learning_seconds, 1e-9)
        ),
        "gradient_updates_per_second": float(
            measurement_vector_steps * gradient_steps
            / max(learning_seconds, 1e-9)
        ),
        "evaluation_frames_per_second": float(evaluation_fps),
        "checkpoint_seconds": float(checkpoint_seconds),
        "learning_measurement_seconds": float(learning_seconds),
        "mean_loss": float(np.mean(losses)),
        "worker_transition_counts": rank_counts,
        "all_workers_produced_transitions": bool(all_workers_produced),
        "active_logical_cores_over_10_percent": int(
            np.sum(mean_per_core > 10.0)
        ),
        "mean_per_core_cpu_percent": [float(value) for value in mean_per_core],
        "lingering_child_process_ids": lingering,
        "stable": bool(all_workers_produced and not lingering),
    }


def estimate_candidate_runtimes(selected_result):
    learning_tps = selected_result["learning_transitions_per_second"]
    evaluation_fps = selected_result["evaluation_frames_per_second"]
    checkpoint_seconds = selected_result["checkpoint_seconds"]
    estimates = {}
    for budget in (10_000_000, 20_000_000, 25_000_000):
        evaluation_events = budget // EVAL_INTERVAL
        estimated_evaluation_frames = 0.0
        projected_limit = 1000
        for event_index in range(evaluation_events):
            if event_index and event_index % 4 == 0:
                projected_limit += 1000
            # Conservative scenario: one slot reaches the active limit,
            # four other slots each survive for roughly 500 pipes.
            estimated_evaluation_frames += 45.0 * (
                projected_limit + 4 * min(projected_limit, 500)
            )
        training_seconds = budget / max(learning_tps, 1e-9)
        evaluation_seconds = (
            estimated_evaluation_frames / max(evaluation_fps, 1e-9)
        )
        checkpoint_overhead = evaluation_events * checkpoint_seconds
        final_limit = max(10_000, projected_limit)
        final_frames = 45.0 * (final_limit + 4 * 1000)
        final_evaluation_seconds = final_frames / max(evaluation_fps, 1e-9)
        plots_export_gif_seconds = 300.0
        raw_seconds = (
            training_seconds
            + evaluation_seconds
            + checkpoint_overhead
            + final_evaluation_seconds
            + plots_export_gif_seconds
        )
        conservative_seconds = raw_seconds * 1.25
        estimates[budget] = {
            "training_hours": training_seconds / 3600,
            "periodic_evaluation_hours": evaluation_seconds / 3600,
            "checkpoint_hours": checkpoint_overhead / 3600,
            "final_and_postprocessing_hours": (
                final_evaluation_seconds + plots_export_gif_seconds
            )
            / 3600,
            "conservative_total_hours": conservative_seconds / 3600,
            "safety_multiplier": 1.25,
        }
    return estimates


CONFIGURATION_BENCHMARK_RESULT = None
if RUN_MODE == "benchmark":
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    results = []
    for n_envs, torch_threads in BENCHMARK_CONFIGURATIONS:
        print(f"Benchmarking {n_envs} workers / {torch_threads} torch threads...")
        result = benchmark_configuration(n_envs, torch_threads)
        results.append(result)
        print(
            f"  stable={result['stable']} | "
            f"learning={result['learning_transitions_per_second']:.1f} t/s | "
            f"active cores={result['active_logical_cores_over_10_percent']}"
        )
    stable_results = [result for result in results if result["stable"]]
    if not stable_results:
        raise RuntimeError("No process-based benchmark configuration was stable.")
    selected_result = max(
        stable_results,
        key=lambda item: item["learning_transitions_per_second"],
    )
    estimates = estimate_candidate_runtimes(selected_result)
    safe_budgets = [
        budget
        for budget, estimate in estimates.items()
        if estimate["conservative_total_hours"] <= 3.5
    ]
    selected_budget = max(safe_budgets) if safe_budgets else None
    CONFIGURATION_BENCHMARK_RESULT = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "python": sys.version,
        "cpu": {
            "physical_cores": PHYSICAL_CORES,
            "logical_threads": LOGICAL_THREADS,
        },
        "learning_starts": LEARNING_STARTS,
        "configurations": results,
        "selected_configuration": selected_result,
        "runtime_estimates": estimates,
        "selected_budget": selected_budget,
        "selection_rule": (
            "Fastest stable measured learning throughput, then largest listed "
            "budget with conservative estimate <= 3.5 hours."
        ),
    }
    with (RUN_DIR / "runtime_benchmark_report.json").open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(CONFIGURATION_BENCHMARK_RESULT, file, indent=2)
    with (RUN_DIR / "selected_configuration.json").open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(
            {
                "selected_configuration": selected_result,
                "runtime_estimates": estimates,
                "selected_budget": selected_budget,
            },
            file,
            indent=2,
        )
    print(json.dumps(CONFIGURATION_BENCHMARK_RESULT, indent=2))

Benchmarking 12 workers / 1 torch threads...

  stable=True | learning=31.2 t/s | active cores=16
Benchmarking 12 workers / 2 torch threads...


  stable=True | learning=34.3 t/s | active cores=17
Benchmarking 12 workers / 4 torch threads...


  stable=True | learning=37.0 t/s | active cores=13
Benchmarking 16 workers / 1 torch threads...


  stable=True | learning=34.0 t/s | active cores=13
Benchmarking 16 workers / 2 torch threads...


  stable=True | learning=33.1 t/s | active cores=15
Benchmarking 20 workers / 1 torch threads...


  stable=True | learning=31.4 t/s | active cores=15
Benchmarking 20 workers / 2 torch threads...


  stable=True | learning=37.8 t/s | active cores=12
{
  "created_at": "2026-07-26T11:49:17",
  "python": "3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]",
  "cpu": {
    "physical_cores": 14,
    "logical_threads": 20
  },
  "learning_starts": 20000,
  "configurations": [
    {
      "n_envs": 12,
      "torch_threads": 1,
      "gradient_steps": 4,
      "approx_transitions_per_update": 3.0,
      "environment_transitions_per_second": 3354.0970575039814,
      "learning_transitions_per_second": 31.188858511502332,
      "gradient_updates_per_second": 10.396286170500778,
      "evaluation_frames_per_second": 861.8605884305484,
      "checkpoint_seconds": 0.05586089999997057,
      "learning_measurement_seconds": 69.25550029999977,
      "mean_loss": 0.10650952413626429,
      "worker_transition_counts": {
        "0": 1947,
        "1": 1947,
        "2": 1947,
        "3": 1947,
        "4": 1947,
        "5": 1947,
        "6": 1947,
        "7": 194

## 8. Integrated smoke checks

These assertions exercise replay insertion/PER sampling, DDQN backpropagation, model serialization, checkpoint resume metadata, five-slot evaluation, holdout isolation, and frame-limit scaling without starting long training.

In [8]:
def run_integrated_smoke_checks():
    env = make_evaluation_env(score_limit=2, render_mode=None)
    try:
        observation, _ = env.reset(seed=SMOKE_EVALUATION_SEEDS[0])
        state_dim = int(env.observation_space.shape[0])
        action_dim = int(env.action_space.n)
        replay = PrioritizedReplayBuffer(1024, state_dim, PER_ALPHA)
        current = observation
        for index in range(BATCH_SIZE + 32):
            action = int(index % action_dim)
            next_observation, reward, terminated, truncated, _ = env.step(action)
            replay.add(
                current,
                action,
                reward,
                next_observation,
                terminated or truncated,
            )
            if terminated or truncated:
                next_observation, _ = env.reset(
                    seed=SMOKE_EVALUATION_SEEDS[0] + index + 1
                )
            current = next_observation
    finally:
        env.close()

    online, target, optimizer = create_networks_and_optimizer(
        state_dim, action_dim
    )
    before = {
        name: tensor.detach().clone()
        for name, tensor in online.state_dict().items()
    }
    metrics = optimize_double_dqn(
        online, target, optimizer, replay, BATCH_SIZE, GAMMA, 0.4
    )
    changed = any(
        not torch.equal(before[name], tensor)
        for name, tensor in online.state_dict().items()
    )
    assert changed and math.isfinite(metrics["loss"])

    state = initial_evaluation_state()
    buffer = io.BytesIO()
    payload = {
        "online_state_dict": online.state_dict(),
        "target_state_dict": target.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "global_step": 250_000,
        "evaluation_state": state,
        "best_ranking_key": [1000, 1, 10, 25.0, 50],
    }
    torch.save(payload, buffer)
    buffer.seek(0)
    restored = torch.load(buffer, map_location=DEVICE, weights_only=False)
    assert restored["evaluation_state"] == state
    assert restored["global_step"] == 250_000

    tracked_non_holdout = (
        set(range(SEED, SEED + SELECTED_N_ENVS))
        | set(initial_evaluation_state()["used_eval_seeds"])
        | set(SMOKE_EVALUATION_SEEDS)
        | {BENCHMARK_SEED, GIF_SEED, DEFAULT_LIVE_SEED}
    )
    assert set(FINAL_HOLDOUT_SEEDS).isdisjoint(tracked_non_holdout)
    assert scaled_frame_safety_limit(10_000) >= 500_000

    evaluation_smoke = evaluate_dqn(
        online,
        seeds=SMOKE_EVALUATION_SEEDS,
        score_limit=1,
        max_steps=2_000,
    )
    assert len(evaluation_smoke["episodes"]) == 5
    return {
        "replay_insertion": "passed",
        "per_sampling": "passed",
        "ddqn_optimization": "passed",
        "model_serialization": "passed",
        "resume_metadata": "passed",
        "five_slot_evaluation": "passed",
        "final_holdout_unseen": "passed",
        "scaled_final_frame_limit": scaled_frame_safety_limit(10_000),
        "evaluation_smoke_scores": evaluation_smoke["scores"],
    }


INTEGRATED_SMOKE_RESULTS = run_integrated_smoke_checks()
print(json.dumps(INTEGRATED_SMOKE_RESULTS, indent=2))

{
  "replay_insertion": "passed",
  "per_sampling": "passed",
  "ddqn_optimization": "passed",
  "model_serialization": "passed",
  "resume_metadata": "passed",
  "five_slot_evaluation": "passed",
  "final_holdout_unseen": "passed",
  "scaled_final_frame_limit": 505000,
  "evaluation_smoke_scores": [
    0,
    0,
    0,
    0,
    0
  ]
}


## 9. Training

Workers are created with explicit `spawn`. Every exit path closes them. Scheduled evaluation/checkpoint boundaries remain exact multiples of 250,000; because vector steps arrive in batches, `actual_training_step` may exceed the scheduled boundary by at most `N_ENVS - 1`, and both values are recorded.

To resume, set `RESUME_CHECKPOINT` above. Weights, optimizer, schedules, and evaluation rotation state resume exactly; replay restarts empty and must warm for another 20,000 collected transitions.

In [9]:
def save_training_history(run_dir, history):
    np.savez(
        run_dir / "training_history.npz",
        episode_scores=np.asarray(history["episode_scores"], dtype=np.int32),
        episode_rewards=np.asarray(history["episode_rewards"], dtype=np.float32),
        episode_lengths=np.asarray(history["episode_lengths"], dtype=np.int32),
        losses=np.asarray(history["losses"], dtype=np.float32),
        mean_q=np.asarray(history["mean_q"], dtype=np.float32),
        mean_abs_td_error=np.asarray(history["mean_abs_td_error"], dtype=np.float32),
        training_steps=np.asarray(history["training_steps"], dtype=np.int64),
    )


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2)
    os.replace(temporary, path)


def run_training():
    global RUN_DIR, CHECKPOINT_DIR

    if RESUME_CHECKPOINT is not None:
        resume_path = Path(RESUME_CHECKPOINT).expanduser().resolve()
        if resume_path.parent.name != "checkpoints":
            raise ValueError("Resume checkpoint must be inside a run checkpoints directory.")
        RUN_DIR = resume_path.parent.parent
        CHECKPOINT_DIR = resume_path.parent
        if not RUN_DIR.is_relative_to(ARTIFACTS_ROOT):
            raise ValueError("Resume checkpoint must be under this repository's artifacts directory.")
    else:
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=False)

    selected_configuration_report = {
        "n_envs": SELECTED_N_ENVS,
        "torch_threads": SELECTED_TORCH_THREADS,
        "gradient_steps": GRADIENT_STEPS,
        "approx_transitions_per_update": APPROX_TRANSITIONS_PER_UPDATE,
        "measured_learning_transitions_per_second": MEASURED_LEARNING_TPS,
        "runtime_estimates_hours": RUNTIME_ESTIMATES_HOURS,
        "selected_total_timesteps": TOTAL_TIMESTEPS,
        "conservative_selected_runtime_hours": CONSERVATIVE_SELECTED_RUNTIME_HOURS,
    }
    write_json(RUN_DIR / "selected_configuration.json", selected_configuration_report)
    write_json(
        RUN_DIR / "runtime_benchmark_report.json",
        {
            "summary": selected_configuration_report,
            "note": (
                "Compact baked summary from the implementation benchmark. "
                "Run FLAPPY_RUN_MODE=benchmark to generate full per-configuration data."
            ),
        },
    )

    training_environment = None
    history = {
        "episode_scores": [],
        "episode_rewards": [],
        "episode_lengths": [],
        "losses": [],
        "mean_q": [],
        "mean_abs_td_error": [],
        "training_steps": [],
    }
    evaluation_history = []
    evaluation_state = initial_evaluation_state()
    best_ranking_key = (-1, -1, -1, -1.0, -1)
    best_checkpoint_step = None
    latest_evaluation = None
    global_step = 0
    next_evaluation_step = EVAL_INTERVAL
    next_checkpoint_step = CHECKPOINT_INTERVAL
    next_target_update_step = TARGET_UPDATE_INTERVAL
    resumed_without_replay = False
    training_completed = False
    interrupted = False

    try:
        training_environment = create_training_vec_env(
            n_envs=SELECTED_N_ENVS,
            base_seed=SEED,
            score_limit=TRAINING_SCORE_LIMIT,
            pipe_gap=PIPE_GAP,
            torch_threads=SELECTED_TORCH_THREADS,
        )
        observations = training_environment.reset()
        state_dim = int(training_environment.observation_space.shape[0])
        action_dim = int(training_environment.action_space.n)
        model_config = build_model_config(state_dim, action_dim)
        online, target, optimizer = create_networks_and_optimizer(
            state_dim, action_dim
        )

        if RESUME_CHECKPOINT is not None:
            checkpoint = load_torch_payload(Path(RESUME_CHECKPOINT))
            online.load_state_dict(checkpoint["online_state_dict"])
            target.load_state_dict(checkpoint["target_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            global_step = int(checkpoint["global_step"])
            evaluation_state = copy.deepcopy(checkpoint["evaluation_state"])
            validate_evaluation_state(evaluation_state)
            best_ranking_key = tuple(checkpoint["best_ranking_key"])
            best_checkpoint_step = checkpoint.get("best_checkpoint_step")
            latest_evaluation = checkpoint.get("latest_evaluation")
            scheduled = checkpoint["next_scheduled_steps"]
            next_evaluation_step = int(scheduled["evaluation"])
            next_checkpoint_step = int(scheduled["checkpoint"])
            next_target_update_step = int(scheduled["target_update"])
            resumed_without_replay = True

        replay = PrioritizedReplayBuffer(BUFFER_SIZE, state_dim, PER_ALPHA)
        replay_learning_start_step = global_step + LEARNING_STARTS
        vector_step = 0
        best_training_score = -1
        progress = tqdm(
            total=RUN_TIMESTEPS,
            initial=min(global_step, RUN_TIMESTEPS),
            desc="Local Dueling DDQN",
            unit="transition",
            dynamic_ncols=True,
        )
        training_started = time.perf_counter()

        try:
            while global_step < RUN_TIMESTEPS:
                epsilon = linear_schedule(
                    global_step,
                    EPSILON_START,
                    EPSILON_END,
                    EPSILON_DECAY_STEPS,
                )
                if global_step < replay_learning_start_step:
                    actions = (
                        np.random.random(SELECTED_N_ENVS)
                        < EXPLORATION_FLAP_PROBABILITY
                    ).astype(np.int64)
                else:
                    actions = online.choose_actions(
                        observations,
                        epsilon=epsilon,
                        exploration_flap_probability=EXPLORATION_FLAP_PROBABILITY,
                    )

                next_observations, rewards, dones, infos = training_environment.step(
                    actions
                )
                replay_next = next_observations.copy()
                for index, done in enumerate(dones):
                    if done and "terminal_observation" in infos[index]:
                        replay_next[index] = infos[index]["terminal_observation"]
                replay.add_batch(
                    observations, actions, rewards, replay_next, dones
                )

                for index, done in enumerate(dones):
                    if not done:
                        continue
                    episode = infos[index].get("episode", {})
                    score = int(episode.get("score", infos[index].get("score", 0)))
                    history["episode_scores"].append(score)
                    history["episode_rewards"].append(float(episode.get("r", np.nan)))
                    history["episode_lengths"].append(int(episode.get("l", 0)))
                    best_training_score = max(best_training_score, score)

                observations = next_observations
                previous_step = global_step
                global_step += SELECTED_N_ENVS
                vector_step += 1
                progress.update(min(global_step, RUN_TIMESTEPS) - previous_step)

                if (
                    global_step >= replay_learning_start_step
                    and len(replay) >= BATCH_SIZE
                    and vector_step % TRAIN_EVERY_VECTOR_STEPS == 0
                ):
                    beta = linear_schedule(
                        global_step,
                        PER_BETA_START,
                        PER_BETA_END,
                        PER_BETA_STEPS,
                    )
                    for _ in range(GRADIENT_STEPS):
                        metrics = optimize_double_dqn(
                            online,
                            target,
                            optimizer,
                            replay,
                            BATCH_SIZE,
                            GAMMA,
                            beta,
                        )
                        history["losses"].append(metrics["loss"])
                        history["mean_q"].append(metrics["mean_q"])
                        history["mean_abs_td_error"].append(
                            metrics["mean_abs_td_error"]
                        )
                        history["training_steps"].append(global_step)

                if global_step >= next_target_update_step:
                    target.load_state_dict(online.state_dict())
                    while next_target_update_step <= global_step:
                        next_target_update_step += TARGET_UPDATE_INTERVAL

                if RUN_MODE == "full" and global_step >= next_evaluation_step:
                    state_before = copy.deepcopy(evaluation_state)
                    evaluation = evaluate_dqn(
                        online,
                        seeds=state_before["active_eval_seeds"],
                        score_limit=state_before["evaluation_score_limit"],
                    )
                    transition = advance_evaluation_state(
                        state_before, evaluation["scores"]
                    )
                    evaluation.update(transition)
                    evaluation["scheduled_training_step"] = int(
                        next_evaluation_step
                    )
                    evaluation["actual_training_step"] = int(global_step)
                    evaluation["ranking_key"] = list(
                        checkpoint_ranking_key(evaluation)
                    )
                    evaluation_history.append(evaluation)
                    latest_evaluation = evaluation
                    evaluation_state = copy.deepcopy(evaluation["state_after"])
                    current_key = tuple(evaluation["ranking_key"])
                    if current_key > best_ranking_key:
                        best_ranking_key = current_key
                        best_checkpoint_step = global_step
                        save_training_checkpoint(
                            RUN_DIR / "best_model.pt",
                            online,
                            target,
                            optimizer,
                            global_step,
                            model_config,
                            evaluation_state,
                            best_ranking_key,
                            best_checkpoint_step,
                            latest_evaluation,
                            next_evaluation_step + EVAL_INTERVAL,
                            next_checkpoint_step,
                            next_target_update_step,
                            "training",
                        )
                    print(
                        f"Evaluation scheduled {next_evaluation_step:,}; "
                        f"actual {global_step:,}; scores={evaluation['scores']}; "
                        f"limit={evaluation['score_limit']}; "
                        f"next seeds={evaluation_state['active_eval_seeds']}"
                    )
                    while next_evaluation_step <= global_step:
                        next_evaluation_step += EVAL_INTERVAL

                if RUN_MODE == "full" and global_step >= next_checkpoint_step:
                    checkpoint_path = CHECKPOINT_DIR / f"ddqn_{global_step:09d}.pt"
                    save_training_checkpoint(
                        checkpoint_path,
                        online,
                        target,
                        optimizer,
                        global_step,
                        model_config,
                        evaluation_state,
                        best_ranking_key,
                        best_checkpoint_step,
                        latest_evaluation,
                        next_evaluation_step,
                        next_checkpoint_step + CHECKPOINT_INTERVAL,
                        next_target_update_step,
                        "training",
                    )
                    while next_checkpoint_step <= global_step:
                        next_checkpoint_step += CHECKPOINT_INTERVAL

                if vector_step % 500 == 0:
                    progress.set_postfix(
                        epsilon=f"{epsilon:.3f}",
                        replay=f"{len(replay):,}",
                        best=best_training_score,
                        loss=(
                            f"{history['losses'][-1]:.4f}"
                            if history["losses"]
                            else "warmup"
                        ),
                    )

            training_completed = True
        except KeyboardInterrupt:
            interrupted = True
            interrupt_path = CHECKPOINT_DIR / f"interrupted_{global_step:09d}.pt"
            save_training_checkpoint(
                interrupt_path,
                online,
                target,
                optimizer,
                global_step,
                model_config,
                evaluation_state,
                best_ranking_key,
                best_checkpoint_step,
                latest_evaluation,
                next_evaluation_step,
                next_checkpoint_step,
                next_target_update_step,
                "interrupted",
            )
            print(f"Training interrupted; resumable checkpoint: {interrupt_path}")
        finally:
            progress.close()

        training_seconds = time.perf_counter() - training_started
        if training_completed:
            save_training_checkpoint(
                RUN_DIR / "final_model.pt",
                online,
                target,
                optimizer,
                global_step,
                model_config,
                evaluation_state,
                best_ranking_key,
                best_checkpoint_step,
                latest_evaluation,
                next_evaluation_step,
                next_checkpoint_step,
                next_target_update_step,
                "completed",
            )
        save_training_history(RUN_DIR, history)
        write_json(RUN_DIR / "evaluation_history.json", evaluation_history)
        return {
            "training_completed": training_completed,
            "interrupted": interrupted,
            "global_step": int(global_step),
            "training_seconds": float(training_seconds),
            "history": history,
            "evaluation_history": evaluation_history,
            "evaluation_state": evaluation_state,
            "best_ranking_key": best_ranking_key,
            "best_checkpoint_step": best_checkpoint_step,
            "best_training_score": best_training_score,
            "model_config": model_config,
            "resumed_without_replay": resumed_without_replay,
        }
    finally:
        close_environments([training_environment])


TRAINING_RESULT = None
if RUN_MODE in {"full", "smoke"}:
    TRAINING_RESULT = run_training()
    print(
        f"Training cell finished: completed={TRAINING_RESULT['training_completed']} | "
        f"transitions={TRAINING_RESULT['global_step']:,} | "
        f"elapsed={TRAINING_RESULT['training_seconds'] / 60:.2f} minutes"
    )

## 10. Final unseen-seed holdout and export

After completed training, the selected best checkpoint is loaded once and evaluated on five seeds from a separate high-numbered range. These holdout results never affect model selection. The score limit is at least 10,000 and the safety-frame cap scales with it.

In [10]:
FINAL_EVALUATION = None
EXPORTED_MODEL_PATH = None
SELECTED_MODEL_PATH = None

if TRAINING_RESULT and TRAINING_RESULT["training_completed"]:
    best_path = RUN_DIR / "best_model.pt"
    final_path = RUN_DIR / "final_model.pt"
    SELECTED_MODEL_PATH = best_path if best_path.exists() else final_path
    selected_model, selected_checkpoint = load_model_from_checkpoint(
        SELECTED_MODEL_PATH
    )

    tracked_seed_sets = {
        "training": set(range(SEED, SEED + SELECTED_N_ENVS)),
        "periodic_evaluation": set(
            TRAINING_RESULT["evaluation_state"]["used_eval_seeds"]
        ),
        "smoke": set(SMOKE_EVALUATION_SEEDS),
        "benchmark": {BENCHMARK_SEED},
        "gif": {GIF_SEED},
        "live_default": {DEFAULT_LIVE_SEED},
    }
    all_non_holdout = set().union(*tracked_seed_sets.values())
    overlap = set(FINAL_HOLDOUT_SEEDS) & all_non_holdout
    if overlap:
        raise RuntimeError(f"Final holdout seed collision: {sorted(overlap)}")

    final_score_limit = max(
        10_000,
        int(TRAINING_RESULT["evaluation_state"]["evaluation_score_limit"]),
    )
    FINAL_EVALUATION = evaluate_dqn(
        selected_model,
        seeds=FINAL_HOLDOUT_SEEDS,
        score_limit=final_score_limit,
        max_steps=scaled_frame_safety_limit(final_score_limit),
    )
    FINAL_EVALUATION["success_percentage"] = (
        100.0
        * FINAL_EVALUATION["reached_limit_count"]
        / len(FINAL_HOLDOUT_SEEDS)
    )
    FINAL_EVALUATION["purpose"] = (
        "One-time final holdout; never used for checkpoint selection."
    )
    write_json(RUN_DIR / "final_evaluation.json", FINAL_EVALUATION)

    EXPORTED_MODEL_PATH = RUN_DIR / "exported_model.pt"
    shutil.copy2(SELECTED_MODEL_PATH, EXPORTED_MODEL_PATH)

    metadata = {
        "algorithm": "Dueling Double DQN with Prioritized Experience Replay",
        "run_mode": RUN_MODE,
        "run_directory": str(RUN_DIR.relative_to(PROJECT_ROOT)),
        "selected_source_model": str(SELECTED_MODEL_PATH.relative_to(PROJECT_ROOT)),
        "exported_model": str(EXPORTED_MODEL_PATH.relative_to(PROJECT_ROOT)),
        "training": {
            "selected_budget": TOTAL_TIMESTEPS,
            "actual_transitions": TRAINING_RESULT["global_step"],
            "duration_seconds": TRAINING_RESULT["training_seconds"],
            "best_checkpoint_step": TRAINING_RESULT["best_checkpoint_step"],
            "best_ranking_key": list(TRAINING_RESULT["best_ranking_key"]),
            "resumed_without_replay": TRAINING_RESULT["resumed_without_replay"],
        },
        "model_config": TRAINING_RESULT["model_config"],
        "evaluation_state": TRAINING_RESULT["evaluation_state"],
        "periodic_evaluation_history": TRAINING_RESULT["evaluation_history"],
        "final_unseen_holdout": FINAL_EVALUATION,
    }
    write_json(RUN_DIR / "metadata.json", metadata)
    print(json.dumps(FINAL_EVALUATION, indent=2))
    print(f"Exported model: {EXPORTED_MODEL_PATH}")

## 11. Training and evaluation plots

In [11]:
def moving_average(values, window=100):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return np.asarray([]), np.asarray([])
    window = min(int(window), len(values))
    kernel = np.ones(window, dtype=np.float32) / window
    return np.arange(window - 1, len(values)), np.convolve(
        values, kernel, mode="valid"
    )


def save_line_plot(path, values, ylabel, title, smoothed=True):
    values = np.asarray(values, dtype=np.float32)
    figure, axis = plt.subplots(figsize=(12, 4))
    if values.size:
        axis.plot(values, alpha=0.25, label="Raw")
        if smoothed:
            x_values, mean = moving_average(values)
            axis.plot(x_values, mean, linewidth=2, label="100-item mean")
            axis.legend()
    else:
        axis.text(0.5, 0.5, "No data recorded", ha="center", va="center")
    axis.set_xlabel("Recorded item")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(alpha=0.25)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.show()
    plt.close(figure)


if TRAINING_RESULT and TRAINING_RESULT["training_completed"]:
    history = TRAINING_RESULT["history"]
    save_line_plot(
        RUN_DIR / "training_scores.png",
        history["episode_scores"],
        "True pipe score",
        "Training episode scores",
    )
    save_line_plot(
        RUN_DIR / "episode_rewards.png",
        history["episode_rewards"],
        "Shaped reward",
        "Training episode rewards",
    )
    save_line_plot(
        RUN_DIR / "episode_lengths.png",
        history["episode_lengths"],
        "Frames",
        "Training episode lengths",
    )
    save_line_plot(
        RUN_DIR / "training_loss.png",
        history["losses"],
        "Prioritized Huber loss",
        "DDQN optimization loss",
        smoothed=False,
    )

    evaluations = TRAINING_RESULT["evaluation_history"]
    figure, axis = plt.subplots(figsize=(11, 4))
    if evaluations:
        steps = [item["scheduled_training_step"] for item in evaluations]
        for label, key in (
            ("Minimum", "minimum_score"),
            ("Mean", "mean_score"),
            ("Maximum", "maximum_score"),
        ):
            axis.plot(
                steps,
                [item[key] for item in evaluations],
                marker="o",
                label=label,
            )
        axis.legend()
    else:
        axis.text(0.5, 0.5, "No periodic evaluations in this run", ha="center")
    axis.set_xlabel("Scheduled training transitions")
    axis.set_ylabel("Greedy true pipe score")
    axis.set_title("Dynamic five-slot periodic evaluation")
    axis.grid(alpha=0.25)
    figure.tight_layout()
    figure.savefig(RUN_DIR / "evaluation_scores.png", dpi=150)
    plt.show()
    plt.close(figure)

## 12. Demonstration GIF

Model decisions still occur every environment frame. Only captured display frames are thinned. The overlay includes model name, score, and seed, and one modest resize/compression pass is used only if the GIF exceeds the transfer-friendly size target.

In [12]:
GIF_PATH = None
GIF_TARGET_SCORE = 100
GIF_MAX_STEPS = 10_000
GIF_CAPTURE_EVERY = 4
GIF_DISPLAY_FPS = 8
GIF_MAX_SIZE_MB = 20.0


def annotate_gif_frame(frame, model_name, score, seed):
    image = PILImage.fromarray(frame)
    draw = ImageDraw.Draw(image)
    draw.rectangle((6, 6, 210, 66), fill=(0, 0, 0))
    draw.text((12, 10), model_name, fill=(255, 255, 255))
    draw.text((12, 29), f"Score: {score}", fill=(255, 255, 255))
    draw.text((12, 48), f"Seed: {seed}", fill=(255, 255, 255))
    return np.asarray(image)


if EXPORTED_MODEL_PATH is not None:
    gif_model, _ = load_model_from_checkpoint(EXPORTED_MODEL_PATH)
    gif_env = make_evaluation_env(GIF_TARGET_SCORE, render_mode="rgb_array")
    frames = []
    score = 0
    try:
        observation, info = gif_env.reset(seed=GIF_SEED)
        score = int(info.get("score", 0))
        frames.append(
            annotate_gif_frame(
                gif_env.render(), "Dueling DDQN", score, GIF_SEED
            )
        )
        for gif_step in range(1, GIF_MAX_STEPS + 1):
            action = int(
                gif_model.choose_actions(
                    np.expand_dims(observation, axis=0), epsilon=0.0
                )[0]
            )
            observation, _, terminated, truncated, info = gif_env.step(action)
            score = int(info.get("score", score))
            if gif_step % GIF_CAPTURE_EVERY == 0 or terminated or truncated:
                frames.append(
                    annotate_gif_frame(
                        gif_env.render(), "Dueling DDQN", score, GIF_SEED
                    )
                )
            if terminated or truncated:
                break
    finally:
        gif_env.close()

    GIF_PATH = RUN_DIR / "demonstration.gif"
    imageio.mimsave(GIF_PATH, frames, fps=GIF_DISPLAY_FPS, loop=0)
    gif_size_mb = GIF_PATH.stat().st_size / (1024**2)
    if gif_size_mb > GIF_MAX_SIZE_MB:
        compressed = []
        for frame in frames[::2]:
            image = PILImage.fromarray(frame)
            image = image.resize(
                (
                    max(1, int(image.width * 0.75)),
                    max(1, int(image.height * 0.75)),
                ),
                PILImage.Resampling.LANCZOS,
            )
            compressed.append(np.asarray(image))
        imageio.mimsave(
            GIF_PATH,
            compressed,
            fps=max(5, GIF_DISPLAY_FPS // 2),
            loop=0,
        )
        frames = compressed
        gif_size_mb = GIF_PATH.stat().st_size / (1024**2)

    dimensions = PILImage.fromarray(frames[0]).size
    print(f"GIF achieved score: {score}")
    print(f"GIF seed: {GIF_SEED}")
    print(f"GIF frames: {len(frames)}")
    print(f"GIF displayed FPS: {GIF_DISPLAY_FPS}")
    print(f"GIF dimensions: {dimensions[0]}x{dimensions[1]}")
    print(f"GIF size: {gif_size_mb:.2f} MB")
    print(f"GIF path: {GIF_PATH}")

## 13. Native local Pygame playback

The playback loop steps the environment and chooses an action on every frame. It first measures every-frame drawing speed. If 30 displayed FPS is sustainable it renders every frame at 30 FPS; otherwise it continues making every decision but displays every second frame at a stable 15 FPS.

Set `RUN_LIVE_PLAYBACK = True` and configure the variables below. Unlimited mode continues until crash, Escape, window close, Ctrl+C, or an explicitly configured safety limit.

In [13]:
RUN_LIVE_PLAYBACK = False
LIVE_MODEL_PATH = EXPORTED_MODEL_PATH
LIVE_SEED = DEFAULT_LIVE_SEED
LIVE_TARGET_SCORE = 1000
LIVE_FPS = 30
LIVE_UNLIMITED = False
LIVE_MAX_STEPS = 60_000  # Set None for no explicit safety limit.


def run_live_playback(
    model_path=LIVE_MODEL_PATH,
    seed=LIVE_SEED,
    target_score=LIVE_TARGET_SCORE,
    requested_fps=LIVE_FPS,
    unlimited=LIVE_UNLIMITED,
    max_steps=LIVE_MAX_STEPS,
):
    if model_path is None or not Path(model_path).exists():
        raise FileNotFoundError(
            "Set LIVE_MODEL_PATH to exported_model.pt or another compatible checkpoint."
        )
    if int(seed) in set(FINAL_HOLDOUT_SEEDS):
        raise ValueError("A reserved final holdout seed cannot be used for playback.")

    model, _ = load_model_from_checkpoint(model_path)
    model_name = Path(model_path).name
    environment_limit = 1_000_000_000 if unlimited else int(target_score)
    env = make_evaluation_env(environment_limit, render_mode="rgb_array")
    pygame.init()
    screen = None
    clock = pygame.time.Clock()
    ended_by = "maximum step safety limit"
    total_reward = 0.0
    live_step = 0
    live_score = 0

    try:
        observation, info = env.reset(seed=int(seed))
        live_score = int(info.get("score", 0))
        first_frame = env.render()
        height, width = first_frame.shape[:2]
        screen = pygame.display.set_mode((width, height))
        pygame.display.set_caption("Flappy Bird - Dueling DDQN")
        font = pygame.font.Font(None, 24)

        benchmark_frames = 120
        benchmark_started = time.perf_counter()
        benchmark_observation = observation
        benchmark_env = make_evaluation_env(environment_limit, render_mode="rgb_array")
        try:
            benchmark_observation, _ = benchmark_env.reset(seed=int(seed))
            for _ in range(benchmark_frames):
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        raise KeyboardInterrupt
                action = int(
                    model.choose_actions(
                        np.expand_dims(benchmark_observation, axis=0), epsilon=0.0
                    )[0]
                )
                benchmark_observation, _, terminated, truncated, _ = (
                    benchmark_env.step(action)
                )
                frame = benchmark_env.render()
                surface = pygame.surfarray.make_surface(
                    np.transpose(frame, (1, 0, 2))
                )
                screen.blit(surface, (0, 0))
                pygame.display.flip()
                if terminated or truncated:
                    benchmark_observation, _ = benchmark_env.reset(seed=int(seed))
        finally:
            benchmark_env.close()

        measured_fps = benchmark_frames / max(
            time.perf_counter() - benchmark_started, 1e-9
        )
        if measured_fps >= 30.0 and requested_fps >= 30:
            display_every = 1
            display_fps = 30
        else:
            display_every = 2
            display_fps = 15

        observation, info = env.reset(seed=int(seed))
        live_score = int(info.get("score", 0))
        step_limit = max_steps if max_steps is not None else sys.maxsize

        for live_step in range(1, int(step_limit) + 1):
            stop_requested = False
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    ended_by = "window closed"
                    stop_requested = True
                elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                    ended_by = "Escape"
                    stop_requested = True
            if stop_requested:
                break

            action = int(
                model.choose_actions(
                    np.expand_dims(observation, axis=0), epsilon=0.0
                )[0]
            )
            observation, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            live_score = int(info.get("score", live_score))

            if live_step % display_every == 0 or terminated or truncated:
                frame = env.render()
                surface = pygame.surfarray.make_surface(
                    np.transpose(frame, (1, 0, 2))
                )
                screen.blit(surface, (0, 0))
                overlay = [
                    f"Model: {model_name}",
                    f"Score: {live_score}",
                    f"Seed: {seed}",
                ]
                for line_index, text in enumerate(overlay):
                    text_surface = font.render(text, True, (255, 255, 255), (0, 0, 0))
                    screen.blit(text_surface, (8, 8 + line_index * 22))
                pygame.display.flip()
                clock.tick(display_fps)

            if terminated:
                ended_by = "bird crashed"
                break
            if truncated:
                ended_by = (
                    "target score reached"
                    if live_score >= environment_limit
                    else "environment truncated"
                )
                break
    except KeyboardInterrupt:
        ended_by = "Ctrl+C or benchmark window close"
    finally:
        env.close()
        pygame.quit()

    summary = {
        "model": str(model_path),
        "seed": int(seed),
        "score": int(live_score),
        "reward": float(total_reward),
        "frames": int(live_step),
        "ended_by": ended_by,
        "measured_unthrottled_display_fps": float(measured_fps),
        "selected_display_fps": int(display_fps),
        "display_every_environment_frames": int(display_every),
        "unlimited": bool(unlimited),
    }
    print(json.dumps(summary, indent=2))
    return summary


LIVE_PLAYBACK_RESULT = None
if RUN_LIVE_PLAYBACK:
    LIVE_PLAYBACK_RESULT = run_live_playback()

## 14. Compact result summary

In [14]:
RESULT_SUMMARY = None
if TRAINING_RESULT and TRAINING_RESULT["training_completed"]:
    periodic = TRAINING_RESULT["evaluation_history"]
    RESULT_SUMMARY = {
        "selected_training_budget": TOTAL_TIMESTEPS,
        "actual_training_transitions": TRAINING_RESULT["global_step"],
        "training_duration_seconds": TRAINING_RESULT["training_seconds"],
        "selected_worker_configuration": {
            "n_envs": SELECTED_N_ENVS,
            "torch_threads": SELECTED_TORCH_THREADS,
            "gradient_steps": GRADIENT_STEPS,
            "approx_transitions_per_update": APPROX_TRANSITIONS_PER_UPDATE,
            "measured_learning_transitions_per_second": MEASURED_LEARNING_TPS,
        },
        "best_checkpoint_step": TRAINING_RESULT["best_checkpoint_step"],
        "best_periodic_ranking_key": list(TRAINING_RESULT["best_ranking_key"]),
        "final_periodic_active_seeds": TRAINING_RESULT["evaluation_state"][
            "active_eval_seeds"
        ],
        "final_periodic_score_limit": TRAINING_RESULT["evaluation_state"][
            "evaluation_score_limit"
        ],
        "final_unseen_holdout_seeds": FINAL_HOLDOUT_SEEDS,
        "final_unseen_holdout_scores": FINAL_EVALUATION["scores"],
        "final_unseen_holdout_statistics": {
            "mean": FINAL_EVALUATION["mean_score"],
            "median": FINAL_EVALUATION["median_score"],
            "std": FINAL_EVALUATION["std_score"],
            "minimum": FINAL_EVALUATION["minimum_score"],
            "maximum": FINAL_EVALUATION["maximum_score"],
        },
        "exported_model_path": str(EXPORTED_MODEL_PATH),
        "gif_path": str(GIF_PATH),
    }
    write_json(RUN_DIR / "result_summary.json", RESULT_SUMMARY)

    if RUN_MODE == "full":
        latest_pointer = ARTIFACTS_ROOT / "latest_run.txt"
        relative_run = RUN_DIR.relative_to(PROJECT_ROOT).as_posix()
        temporary_pointer = latest_pointer.with_suffix(".txt.tmp")
        temporary_pointer.write_text(relative_run + "\n", encoding="utf-8")
        os.replace(temporary_pointer, latest_pointer)

    print(json.dumps(RESULT_SUMMARY, indent=2))
    if RUN_MODE == "smoke":
        print(
            "SMOKE RUN ONLY: these results validate execution and must not be "
            "reported as successful 10M/20M/25M training."
        )
elif RUN_MODE == "benchmark":
    print("Benchmark-only execution complete; no training model was produced.")
else:
    print("Training did not complete; use the interrupted checkpoint to resume.")

Benchmark-only execution complete; no training model was produced.
